# Chapter 9 — System Design and Scalability

System design interviews are conversations, not code. The approach:
1. **Scope the problem**: which features, and how many users?
2. **Make reasonable assumptions** and state them.
3. **Draw the major components.**
4. **Identify the key issues**: bottlenecks and hot spots.
5. **Redesign for those issues.**

Building blocks: horizontal vs. vertical scaling, load balancers, denormalization, **sharding** (vertical, key-based/hash, directory-based), caching, async processing and queues, networking metrics (bandwidth, throughput, latency), and MapReduce.

The notebook turns the core ideas of each question into **runnable toy models**. Treat them as whiteboard sketches that happen to execute.

In [1]:
from __future__ import annotations

import bisect
import hashlib
import heapq
import random
import string
from collections import Counter, OrderedDict, defaultdict, deque
from dataclasses import dataclass, field

### Building block: consistent hashing
Plain `hash(key) % N` remaps almost every key when N changes. A hash ring with virtual nodes moves only ~1/N of the keys.

In [2]:
def h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)


class ConsistentHashRing:
    def __init__(self, nodes: list[str], vnodes: int = 100) -> None:
        self.vnodes = vnodes
        self._ring: list[tuple[int, str]] = []
        for n in nodes:
            self.add(n)

    def add(self, node: str) -> None:
        for i in range(self.vnodes):
            bisect.insort(self._ring, (h(f"{node}#{i}"), node))

    def remove(self, node: str) -> None:
        self._ring = [(k, n) for k, n in self._ring if n != node]

    def node_for(self, key: str) -> str:
        i = bisect.bisect(self._ring, (h(key), "")) % len(self._ring)
        return self._ring[i][1]


keys = [f"user{i}" for i in range(10_000)]
ring = ConsistentHashRing(["db0", "db1", "db2", "db3"])
before = {k: ring.node_for(k) for k in keys}
ring.add("db4")
moved = sum(before[k] != ring.node_for(k) for k in keys) / len(keys)
naive_moved = sum(h(k) % 4 != h(k) % 5 for k in keys) / len(keys)
print(f"consistent hashing moved {moved:.0%} of keys; modulo hashing moved {naive_moved:.0%}")
print("load:", Counter(ring.node_for(k) for k in keys))

consistent hashing moved 19% of keys; modulo hashing moved 80%
load: Counter({'db0': 2261, 'db3': 2231, 'db4': 1907, 'db1': 1873, 'db2': 1728})


### Building block: MapReduce

In [3]:
from itertools import groupby


def map_reduce(docs, mapper, reducer):
    pairs = [kv for d in docs for kv in mapper(d)]              # map
    pairs.sort(key=lambda kv: kv[0])                            # shuffle/sort
    return {k: reducer(k, [v for _, v in grp]) for k, grp in groupby(pairs, key=lambda kv: kv[0])}  # reduce


wc = map_reduce(["a b a", "b c"], lambda d: [(w, 1) for w in d.split()], lambda k, vs: sum(vs))
assert wc == {"a": 2, "b": 2, "c": 1}
print("mapreduce ok")

mapreduce ok


## 9.1 Stock Data
Serve end-of-day stock data (open, close, high, low) to about 1,000 clients. Options: FTP text files, a SQL database, or XML/JSON over HTTP. A reasonable answer is an **HTTP JSON API backed by a database**. It's easy to query, easy to extend, and cacheable. Below is a toy in-memory store with the query surface a REST API would expose.

In [4]:
from datetime import date, timedelta


@dataclass(frozen=True)
class Quote:
    symbol: str
    day: date
    open: float
    close: float
    high: float
    low: float


class StockService:
    def __init__(self) -> None:
        self._by_symbol: dict[str, list[Quote]] = defaultdict(list)  # kept sorted by day

    def ingest(self, q: Quote) -> None:
        bisect.insort(self._by_symbol[q.symbol], q, key=lambda x: x.day)

    def get(self, symbol: str, start: date, end: date) -> list[dict]:
        """GET /quotes/{symbol}?start=...&end=... -> JSON rows."""
        rows = self._by_symbol[symbol]
        lo = bisect.bisect_left(rows, start, key=lambda x: x.day)
        hi = bisect.bisect_right(rows, end, key=lambda x: x.day)
        return [q.__dict__ | {"day": q.day.isoformat()} for q in rows[lo:hi]]


svc = StockService()
d0 = date(2026, 1, 1)
for i in range(30):
    price = 100 + i
    svc.ingest(Quote("ACME", d0 + timedelta(days=i), price, price + 1, price + 2, price - 1))
assert len(svc.get("ACME", d0 + timedelta(5), d0 + timedelta(9))) == 5
print(svc.get("ACME", d0, d0)[0])

{'symbol': 'ACME', 'day': '2026-01-01', 'open': 100, 'close': 101, 'high': 102, 'low': 99}


## 9.2 Social Network
Design the data structures for a very large social network, and an algorithm to find the shortest connection path between two people. On one machine, **bidirectional BFS** visits about `O(k^(q/2))` nodes instead of `O(k^q)`. At scale, each person lives on a machine found by `machine_for(person_id)`. Group lookups by machine, and cache machine lookups.

In [5]:
class Machine:
    def __init__(self, mid: int) -> None:
        self.id = mid
        self.people: dict[int, list[int]] = {}  # person -> friend ids


class Server:
    def __init__(self, n_machines: int) -> None:
        self.machines = [Machine(i) for i in range(n_machines)]
        self.lookups = 0

    def machine_for(self, person: int) -> Machine:
        return self.machines[person % len(self.machines)]

    def friends(self, person: int) -> list[int]:
        self.lookups += 1
        return self.machine_for(person).people.get(person, [])


def bidirectional_bfs(server: Server, src: int, dst: int) -> list[int] | None:
    if src == dst:
        return [src]
    parents = [{src: None}, {dst: None}]
    frontiers = [deque([src]), deque([dst])]
    while frontiers[0] and frontiers[1]:
        side = 0 if len(frontiers[0]) <= len(frontiers[1]) else 1  # expand the smaller side
        for _ in range(len(frontiers[side])):
            person = frontiers[side].popleft()
            for f in server.friends(person):
                if f in parents[side]:
                    continue
                parents[side][f] = person
                if f in parents[1 - side]:  # collision - stitch the path
                    def walk(p: int | None, par: dict) -> list[int]:
                        out = []
                        while p is not None:
                            out.append(p); p = par[p]
                        return out
                    left = walk(f, parents[0])[::-1]
                    right = walk(parents[1][f], parents[1])
                    return left + right
                frontiers[side].append(f)
    return None


random.seed(7)
srv = Server(8)
N = 20_000
adj: dict[int, set[int]] = defaultdict(set)
for a in range(N):
    for b in random.sample(range(N), 5):
        if a != b:
            adj[a].add(b); adj[b].add(a)
for p, fs in adj.items():
    srv.machine_for(p).people[p] = sorted(fs)
path = bidirectional_bfs(srv, 0, 12345)
assert path[0] == 0 and path[-1] == 12345 and all(b in adj[a] for a, b in zip(path, path[1:]))
print("path:", path, "| friend-list lookups:", srv.lookups, "of", N, "people")

path: [0, 1582, 6877, 11094, 7685, 5150, 12345] | friend-list lookups: 98 of 20000 people


## 9.3 Web Crawler
A crawler must avoid infinite loops. URLs alone aren't enough (query parameters, mirrors), so dedupe on a **signature** of (normalized URL + content hash). Revisit pages with a lower priority instead of never, which lets the crawler handle changing content.

In [6]:
from urllib.parse import urldefrag, urlparse, urlunparse


def normalize(url: str) -> str:
    url, _ = urldefrag(url)
    p = urlparse(url)
    return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path.rstrip("/") or "/", "", p.query, ""))


# A toy web: url -> (content, links). Note the cycle a -> b -> a and the duplicate-content mirror.
WEB = {
    "http://x.com/": ("home", ["http://x.com/a", "http://X.com/a#top", "http://mirror.com/"]),
    "http://x.com/a": ("page a", ["http://x.com/b"]),
    "http://x.com/b": ("page b", ["http://x.com/a", "http://x.com/"]),
    "http://mirror.com/": ("home", ["http://x.com/a"]),
}


def crawl(seed: str, max_pages: int = 100) -> list[str]:
    seen_urls: set[str] = set()
    seen_content: set[str] = set()
    frontier: list[tuple[int, str]] = [(0, seed)]  # (priority, url)
    visited: list[str] = []
    while frontier and len(visited) < max_pages:
        _, url = heapq.heappop(frontier)
        url = normalize(url)
        if url in seen_urls or url not in WEB:
            continue
        seen_urls.add(url)
        content, links = WEB[url]
        sig = hashlib.sha1(content.encode()).hexdigest()
        if sig in seen_content:
            continue  # duplicate page
        seen_content.add(sig)
        visited.append(url)
        for link in links:
            heapq.heappush(frontier, (len(visited), link))
    return visited


crawled = crawl("http://x.com/")
print(crawled)
assert crawled == ["http://x.com/", "http://x.com/a", "http://x.com/b"]

['http://x.com/', 'http://x.com/a', 'http://x.com/b']


## 9.4 Duplicate URLs
Find duplicates among 10 billion URLs. That's about 4 TB, which won't fit in memory. **Partition by hash** into files (or machines) so duplicates are guaranteed to land in the same partition, then dedupe each partition in memory. Toy version:

In [7]:
def find_duplicate_urls(urls, partitions: int = 16) -> set[str]:
    buckets: list[list[str]] = [[] for _ in range(partitions)]  # stand-ins for files / machines
    for u in urls:
        buckets[h(u) % partitions].append(u)
    dups: set[str] = set()
    for b in buckets:  # each fits in memory
        seen: set[str] = set()
        for u in b:
            if u in seen:
                dups.add(u)
            seen.add(u)
    return dups


urls = [f"http://site{i % 5000}.com/{i % 777}" for i in range(20_000)]
assert find_duplicate_urls(urls) == {u for u, c in Counter(urls).items() if c > 1}
print("9.4 ok")

9.4 ok


## 9.5 Cache
A search engine's `process_search(query)` is expensive, so design a cache for the most recent queries. Use an **LRU cache** (hash map + doubly linked list; Python's `OrderedDict` provides both). To distribute it, use consistent hashing on the query so each machine owns a slice. Handle staleness with a TTL.

In [8]:
import time


class LRUCacheTTL:
    def __init__(self, capacity: int, ttl: float) -> None:
        self.capacity, self.ttl = capacity, ttl
        self._d: OrderedDict[str, tuple[float, list[str]]] = OrderedDict()

    def get(self, key: str) -> list[str] | None:
        item = self._d.get(key)
        if item is None:
            return None
        ts, val = item
        if time.monotonic() - ts > self.ttl:
            del self._d[key]
            return None
        self._d.move_to_end(key)
        return val

    def put(self, key: str, val: list[str]) -> None:
        self._d[key] = (time.monotonic(), val)
        self._d.move_to_end(key)
        if len(self._d) > self.capacity:
            self._d.popitem(last=False)


class DistributedSearchCache:
    def __init__(self, machines: int, capacity_each: int) -> None:
        names = [f"cache{i}" for i in range(machines)]
        self.ring = ConsistentHashRing(names)
        self.nodes = {n: LRUCacheTTL(capacity_each, ttl=60) for n in names}
        self.backend_calls = 0

    def _process_search(self, q: str) -> list[str]:
        self.backend_calls += 1
        return [f"{q}-result{i}" for i in range(3)]

    def search(self, q: str) -> list[str]:
        node = self.nodes[self.ring.node_for(q.lower().strip())]
        if (hit := node.get(q)) is not None:
            return hit
        res = self._process_search(q)
        node.put(q, res)
        return res


dc = DistributedSearchCache(4, capacity_each=50)
qs = [f"q{random.randint(0, 60)}" for _ in range(2000)]
for q in qs:
    dc.search(q)
print(f"{len(qs)} searches, {dc.backend_calls} hit the backend")

2000 searches, 61 hit the backend


## 9.6 Sales Rank
Show each product's best-seller rank overall and per category, based on the past week's sales. Store **per-day counts** in a ring of 7 buckets so old days drop out cheaply. Recompute the rankings periodically in batch (e.g. hourly), not per purchase. At scale, append purchases to a log, aggregate them with MapReduce, and store rankings in a denormalized table.

In [9]:
class SalesRank:
    def __init__(self) -> None:
        self.daily: dict[str, list[int]] = defaultdict(lambda: [0] * 7)
        self.category: dict[str, set[str]] = defaultdict(set)
        self.today = 0
        self.rankings: dict[str, list[str]] = {}

    def record_sale(self, product: str, categories: list[str], qty: int = 1) -> None:
        self.daily[product][self.today % 7] += qty
        for c in categories + ["ALL"]:
            self.category[c].add(product)

    def advance_day(self) -> None:
        self.today += 1
        for counts in self.daily.values():
            counts[self.today % 7] = 0  # the bucket from 7 days ago is reused

    def recompute(self) -> None:  # the periodic batch job
        weekly = {p: sum(c) for p, c in self.daily.items()}
        self.rankings = {cat: sorted(ps, key=lambda p: -weekly[p]) for cat, ps in self.category.items()}

    def rank(self, product: str, category: str = "ALL") -> int:
        return self.rankings[category].index(product) + 1


sr = SalesRank()
sr.record_sale("soap", ["home"], 5); sr.record_sale("tv", ["electronics", "home"], 3); sr.record_sale("phone", ["electronics"], 4)
for _ in range(6):
    sr.advance_day()
sr.record_sale("tv", ["electronics", "home"], 3)
sr.recompute()
assert sr.rank("tv") == 1 and sr.rank("phone", "electronics") == 2
sr.advance_day(); sr.recompute()  # day 0's sales expire
assert sr.rank("tv") == 1 and sr.rank("soap", "home") == 2
print("9.6 ok")

9.6 ok


## 9.7 Personal Financial Manager
The service connects to bank accounts, analyzes spending, and makes recommendations. Components: bank-data pulls (async, via a task queue), a transactions table, a **categorizer** based on seller rules plus user overrides, budgets, and batched email alerts. Transactions are append-only, so categorizing them is a natural stream job.

In [10]:
@dataclass
class Transaction:
    user: str
    seller: str
    amount: float


SELLER_CATEGORY = {"grocer": "food", "cafe": "food", "airline": "travel", "cinema": "entertainment"}


class FinanceManager:
    def __init__(self) -> None:
        self.overrides: dict[tuple[str, str], str] = {}
        self.budgets: dict[tuple[str, str], float] = {}
        self.spent: dict[tuple[str, str], float] = defaultdict(float)
        self.queue: deque[Transaction] = deque()  # async ingestion
        self.alerts: list[str] = []

    def categorize(self, t: Transaction) -> str:
        return self.overrides.get((t.user, t.seller)) or SELLER_CATEGORY.get(t.seller, "other")

    def ingest(self, t: Transaction) -> None:
        self.queue.append(t)

    def process(self) -> None:
        while self.queue:
            t = self.queue.popleft()
            key = (t.user, self.categorize(t))
            before = self.spent[key]
            self.spent[key] += t.amount
            budget = self.budgets.get(key)
            if budget is not None and before <= budget < self.spent[key]:
                self.alerts.append(f"{t.user}: over {key[1]} budget ({self.spent[key]:.2f} > {budget:.2f})")


fm = FinanceManager()
fm.budgets[("amy", "food")] = 50
fm.overrides[("amy", "cinema")] = "date night"
for s, amt in [("grocer", 30), ("cafe", 25), ("cinema", 20), ("cafe", 5)]:
    fm.ingest(Transaction("amy", s, amt))
fm.process()
assert fm.alerts == ["amy: over food budget (55.00 > 50.00)"]
assert fm.spent[("amy", "date night")] == 20
print(fm.alerts)

['amy: over food budget (55.00 > 50.00)']


## 9.8 Pastebin
Users paste text and get a random URL back, and each document tracks its access count. Key points: generate unguessable IDs (random base62, retrying on collision), keep metadata in a DB and content in a file/object store, cache the popular documents, and aggregate analytics from logs.

In [11]:
import secrets

ALPHABET = string.ascii_letters + string.digits


class Pastebin:
    def __init__(self, key_len: int = 8) -> None:
        self.key_len = key_len
        self.store: dict[str, str] = {}          # object storage
        self.meta: dict[str, dict] = {}          # DB rows
        self.cache = LRUCacheTTL(capacity=100, ttl=300)

    def _new_key(self) -> str:
        while True:
            key = "".join(secrets.choice(ALPHABET) for _ in range(self.key_len))
            if key not in self.store:
                return key

    def create(self, text: str, expires_s: float | None = None) -> str:
        key = self._new_key()
        self.store[key] = text
        self.meta[key] = {"created": time.time(), "expires": expires_s and time.time() + expires_s, "views": 0}
        return f"https://paste.example/{key}"

    def get(self, url: str) -> str | None:
        key = url.rsplit("/", 1)[-1]
        m = self.meta.get(key)
        if m is None or (m["expires"] and time.time() > m["expires"]):
            return None
        m["views"] += 1
        if (cached := self.cache.get(key)) is not None:
            return cached[0]
        text = self.store[key]
        self.cache.put(key, [text])
        return text


pb = Pastebin()
u = pb.create("print('hello')")
assert pb.get(u) == "print('hello')" and pb.get(u) and pb.meta[u.rsplit('/', 1)[1]]["views"] == 2
print(u)

https://paste.example/yVt2zIlC
